# Code mẫu — k-Nearest Neighbors

kNN cần hai quyết định: đo khoảng cách bằng cách nào, và gộp các láng giềng lại theo quy tắc gì. Các ví dụ dưới đây làm rõ hai quyết định ấy bằng tay, trước khi dùng thư viện để chọn k.

## Ví dụ 1: Ba hồ sơ gần nhất bỏ phiếu ra sao?

Năm hồ sơ có hai điểm số trên cùng một thang, và mỗi hồ sơ thuộc nhóm 0 hoặc nhóm 1. Với truy vấn `[2, 2]`, ta tính bình phương khoảng cách, chọn ba láng giềng gần nhất, và in ra cả chỉ số lẫn số phiếu. Bản cài nhỏ này có hai luật phá hòa: nếu khoảng cách bằng nhau thì giữ thứ tự hàng, nếu số phiếu bằng nhau thì chọn nhãn nhỏ hơn.

In [ ]:
import numpy as np

X = np.array([[1., 1.], [2., 1.], [4., 3.], [5., 4.], [3., 2.]])
y = np.array([0, 0, 1, 1, 1])
query = np.array([2., 2.])
distance2 = np.sum((X - query) ** 2, axis=1)
order = np.argsort(distance2, kind="stable")
neighbors = order[:3]
votes = np.bincount(y[neighbors], minlength=2)
print("d²:", distance2.tolist())
print("3 láng giềng:", neighbors.tolist(), "nhãn:", y[neighbors].tolist())
print("phiếu / dự đoán:", votes.tolist(), int(votes.argmax()))
print("k=1:", int(y[order[0]]), "hai điểm cùng gần nhất:", order[:2].tolist())
assert neighbors.tolist() == [1, 4, 0]

d²: [2.0, 1.0, 5.0, 13.0, 1.0]
3 láng giềng: [1, 4, 0] nhãn: [0, 1, 0]
phiếu / dự đoán: [2, 1] 0
k=1: 0 hai điểm cùng gần nhất: [1, 4]


Ba láng giềng là hàng 1, 4 và 0, cho hai phiếu nhóm 0 và một phiếu nhóm 1, nên nhóm 0 thắng. Với k = 1 thì có chuyện: hàng 1 và hàng 4 cùng cách truy vấn đúng 1, mà lại khác nhãn. Kết quả lúc này hoàn toàn do luật phá hòa quyết định. Luật ấy là của bản cài này, các thư viện khác có thể làm khác.

Vẫn dùng láng giềng, nhưng chuyển sang bài hồi quy để phân biệt trọng số `1/d` với `1/d²`. Hai láng giềng có target là 0 và 6, ở khoảng cách thật là 1 và 2.

In [ ]:
distance = np.array([1., 2.])
targets = np.array([0., 6.])
print("đều:", float(targets.mean()))
print("1/d:", round(float(np.average(targets, weights=1 / distance)), 3))
print("1/d²:", round(float(np.average(targets, weights=1 / distance**2)), 3))

đều: 3.0
1/d: 2.0
1/d²: 1.2


Trọng số đều cho 3, trọng số `1/d` cho 2, còn `1/d²` cho 1.2. Bỏ căn để xếp hạng láng giềng thì không sao, vì thứ tự không đổi. Nhưng nếu bỏ căn rồi mới tính trọng số, ta đã vô tình đổi sang một thuật toán khác. Nếu có láng giềng ở khoảng cách 0 thì không được chia thẳng. Phải có luật riêng cho những điểm trùng với truy vấn, giống như thư viện vẫn xử lý riêng trường hợp này.

## Ví dụ 2: Đổi giờ sang phút làm đổi luôn hàng xóm gần nhất

Hai hồ sơ có `(điểm, giờ học)` là `[7, 5]` và `[8, 6]`, còn truy vấn là `[8, 5.2]`. Ta chỉ đổi đơn vị của giờ học sang phút, thông tin thực tế vẫn y nguyên. So khoảng cách thô với khoảng cách sau khi chuẩn hóa bằng StandardScaler học từ hai hồ sơ train.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.array([[7., 5.], [8., 6.]])
query = np.array([[8., 5.2]])
scaled_distances = []
for unit, factor in [("giờ", 1.), ("phút", 60.)]:
    factors = np.array([1., factor])
    raw_distance = np.linalg.norm(X * factors - query * factors, axis=1)
    scaler = StandardScaler().fit(X * factors)
    Z = scaler.transform(X * factors)
    z_query = scaler.transform(query * factors)
    distance = np.linalg.norm(Z - z_query, axis=1)
    scaled_distances.append(distance)
    print(unit, "khoảng cách thô:", raw_distance.round(3).tolist(),
          "gần nhất:", int(raw_distance.argmin()))
    print(unit, "khoảng cách chuẩn hóa:", distance.round(3).tolist())
print("sau chuẩn hóa không đổi:", np.allclose(*scaled_distances))
assert np.allclose(*scaled_distances)

giờ khoảng cách thô: [1.02, 0.8] gần nhất: 1
giờ khoảng cách chuẩn hóa: [2.04, 1.6]
phút khoảng cách thô: [12.042, 48.0] gần nhất: 0
phút khoảng cách chuẩn hóa: [2.04, 1.6]
sau chuẩn hóa không đổi: True


Tính bằng giờ thì hồ sơ 1 gần nhất, tính bằng phút thì lại là hồ sơ 0. Khi đổi sang phút, cột giờ học có số rất lớn, nên nó lấn át hẳn cột điểm trong khoảng cách. Sau khi chuẩn hóa, mean và std của cột cũng đổi theo đơn vị, nên hai vector khoảng cách khớp nhau hoàn toàn. Đây là phép kiểm tính bất biến theo đơn vị. Với chỉ hai mẫu train, thang đo học được chưa chắc đã tốt cho bài thật. Dùng scaler nào, hay gán trọng số riêng cho từng feature, vẫn phải kiểm bằng validation.

## Ví dụ 3: Chọn k cho dữ liệu rượu, trên cùng các fold

Wine là bộ dữ liệu có sẵn trong scikit-learn, gồm 13 phép đo với thang đo rất khác nhau, và ba loại rượu. Ta tách riêng một tập test. Trên phần còn lại, ta so các giá trị k cho hai Pipeline, một có scaler và một không. Dùng chung các fold cho cả hai giúp tách được ảnh hưởng của thang đo khỏi ảnh hưởng của cách chia.

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=10
)
folds = list(StratifiedKFold(3, shuffle=True, random_state=6).split(X_dev, y_dev))
searches = {}
for name, scaler in [("thô", "passthrough"), ("chuẩn hóa", StandardScaler())]:
    pipeline = Pipeline([("scale", scaler), ("knn", KNeighborsClassifier())])
    search = GridSearchCV(pipeline, {"knn__n_neighbors": [1, 3, 5, 9]},
                          cv=folds, scoring="accuracy", n_jobs=1)
    search.fit(X_dev, y_dev)
    searches[name] = search
    print(name, "CV theo k=1,3,5,9:", search.cv_results_["mean_test_score"].round(3).tolist())
    print("k chọn:", search.best_params_["knn__n_neighbors"])

thô CV theo k=1,3,5,9: [0.722, 0.7, 0.729, 0.691]
k chọn: 5
chuẩn hóa CV theo k=1,3,5,9: [0.932, 0.962, 0.947, 0.947]
k chọn: 3


Với thang đo thô, k = 5 được chọn, và CV chỉ khoảng 0.729. Có scaler, k = 3 được chọn, và CV lên khoảng 0.962. Hai dòng dùng đúng cùng những hàng validation, nên khoảng chênh hơn 0.2 này đến từ thang đo, chứ không phải do may mắn khi chia. Trong Wine có cột độ lớn hàng trăm, trong khi cột khác chỉ quanh 1, nên không chuẩn hóa thì khoảng cách gần như chỉ phụ thuộc vào một hai cột. Điểm CV tốt nhất đã được dùng để chọn cấu hình, còn test được giữ lại để đánh giá phương án cuối.

In [ ]:
name = max(searches, key=lambda key: searches[key].best_score_)
chosen = searches[name].best_estimator_
pred = chosen.predict(X_test)
prob = chosen.predict_proba(X_test)
print("chọn bằng CV:", name)
print("test accuracy:", round(accuracy_score(y_test, pred), 3))
print("confusion matrix, hàng=thật, cột=đoán:")
print(confusion_matrix(y_test, pred, labels=[0, 1, 2]))
print("3 hàng xác suất:", prob[:3].round(3).tolist())
assert np.allclose(prob.sum(axis=1), 1.)
assert pred.shape == y_test.shape

chọn bằng CV: chuẩn hóa
test accuracy: 0.956
confusion matrix, hàng=thật, cột=đoán:
[[15  0  0]
 [ 2 16  0]
 [ 0  0 12]]
3 hàng xác suất: [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


Ô ở hàng 1, cột 0 bằng 2, nghĩa là hai mẫu thuộc lớp 1 bị đoán thành lớp 0. Đúng 43 trên 45 mẫu, nên accuracy là 0.956. Ba hàng xác suất đầu chỉ có 0 và 1, vì cả ba láng giềng đều cùng ý kiến, chứ không có nghĩa model chắc chắn tuyệt đối. Với phiếu đều, “xác suất” của kNN chỉ là tỉ lệ nhãn trong k láng giềng, nên với k = 3 nó chỉ nhận vài giá trị như 0, 1/3, 2/3, 1. Khi có ba lớp, k lẻ vẫn có thể hòa, chẳng hạn khi ba láng giềng mang ba nhãn khác nhau.